In [1]:
import tkinter as tk
from tkinter import ttk, messagebox
import random, math, time
from dataclasses import dataclass
from typing import List, Tuple, Optional

# =========================
#   Dots & Boxes (4x4 boxes)
#   - GUI: Tkinter Canvas
#   - AI: Random / MonteCarlo / MCTS (Adversarial UCT)
#   Goal: AI should be strong (at least draw most of the time)
# =========================

N = 4               # 4x4 boxes => 5x5 dots
DOTS = N + 1
AI_PLAYER = 2
HUMAN_PLAYER = 1

# ---------- Game State ----------
@dataclass(frozen=True)
class Move:
    kind: str  # 'H' or 'V'
    r: int
    c: int

@dataclass
class State:
    # H: (N+1) x N  ; V: N x (N+1)
    H: List[List[int]]        # 0 empty, 1 human, 2 ai
    V: List[List[int]]
    boxes: List[List[int]]    # N x N owner (0 none, 1 human, 2 ai)
    turn: int                 # 1 human, 2 ai
    score1: int               # human
    score2: int               # ai

def new_state(first_player=HUMAN_PLAYER) -> State:
    H = [[0 for _ in range(N)] for _ in range(N + 1)]
    V = [[0 for _ in range(N + 1)] for _ in range(N)]
    boxes = [[0 for _ in range(N)] for _ in range(N)]
    return State(H, V, boxes, first_player, 0, 0)

def clone_state(s: State) -> State:
    return State(
        [row[:] for row in s.H],
        [row[:] for row in s.V],
        [row[:] for row in s.boxes],
        s.turn,
        s.score1,
        s.score2
    )

def legal_moves(s: State) -> List[Move]:
    moves = []
    for r in range(N + 1):
        for c in range(N):
            if s.H[r][c] == 0:
                moves.append(Move('H', r, c))
    for r in range(N):
        for c in range(N + 1):
            if s.V[r][c] == 0:
                moves.append(Move('V', r, c))
    return moves

def is_terminal(s: State) -> bool:
    for r in range(N + 1):
        for c in range(N):
            if s.H[r][c] == 0:
                return False
    for r in range(N):
        for c in range(N + 1):
            if s.V[r][c] == 0:
                return False
    return True

def apply_move(s: State, m: Move) -> Tuple[State, bool, int]:
    """
    Returns: (new_state, extra_turn, boxes_completed_count)
    """
    ns = clone_state(s)
    player = ns.turn

    # Place edge
    if m.kind == 'H':
        ns.H[m.r][m.c] = player
    else:
        ns.V[m.r][m.c] = player

    def box_complete(br, bc) -> bool:
        return ns.H[br][bc] and ns.H[br + 1][bc] and ns.V[br][bc] and ns.V[br][bc + 1]

    # Only nearby boxes can be affected
    candidates = []
    if m.kind == 'H':
        r, c = m.r, m.c
        if r > 0: candidates.append((r - 1, c))
        if r < N: candidates.append((r, c))
    else:
        r, c = m.r, m.c
        if c > 0: candidates.append((r, c - 1))
        if c < N: candidates.append((r, c))

    gained = 0
    for br, bc in candidates:
        if 0 <= br < N and 0 <= bc < N and ns.boxes[br][bc] == 0:
            if box_complete(br, bc):
                ns.boxes[br][bc] = player
                gained += 1

    if gained:
        if player == HUMAN_PLAYER:
            ns.score1 += gained
        else:
            ns.score2 += gained
        extra = True
    else:
        extra = False
        ns.turn = HUMAN_PLAYER if player == AI_PLAYER else AI_PLAYER

    return ns, extra, gained

# =========================
#   Heuristics for stronger rollouts
# =========================
def immediate_box_moves(s: State) -> List[Move]:
    im = []
    for m in legal_moves(s):
        ns, _, gained = apply_move(s, m)
        if gained > 0:
            im.append(m)
    return im

def would_create_third_side(s: State, m: Move) -> bool:
    """
    Returns True if making move m causes any unfinished box to become exactly 3-sided (a "gift").
    """
    ns, _, _ = apply_move(s, m)

    def sides(br, bc) -> int:
        cnt = 0
        cnt += 1 if ns.H[br][bc] else 0
        cnt += 1 if ns.H[br+1][bc] else 0
        cnt += 1 if ns.V[br][bc] else 0
        cnt += 1 if ns.V[br][bc+1] else 0
        return cnt

    for br in range(N):
        for bc in range(N):
            if ns.boxes[br][bc] == 0 and sides(br, bc) == 3:
                return True
    return False

def rollout_policy_move(s: State) -> Move:
    """
    Smarter-than-random rollout:
    1) If can complete a box, do it.
    2) Else pick a "safe" move that doesn't create a 3-sided box.
    3) Else random.
    """
    im = immediate_box_moves(s)
    if im:
        return random.choice(im)

    moves = legal_moves(s)
    safe = [m for m in moves if not would_create_third_side(s, m)]
    if safe:
        return random.choice(safe)

    return random.choice(moves)

def rollout_playout(s: State) -> int:
    """
    Play rollout to terminal and return AI perspective reward:
      reward = (AI_score - Human_score)
    """
    rs = clone_state(s)
    while not is_terminal(rs):
        m = rollout_policy_move(rs)
        rs, _, _ = apply_move(rs, m)
    return rs.score2 - rs.score1

# =========================
#   Monte Carlo (flat)
# =========================
def best_move_montecarlo(s: State, sims_per_move=80, time_limit=2.0) -> Move:
    moves = legal_moves(s)
    best_m = moves[0]
    best_val = -10**9
    start = time.time()

    for m in moves:
        total = 0
        done = 0
        while done < sims_per_move:
            ns, _, _ = apply_move(s, m)
            total += rollout_playout(ns)
            done += 1
            if (time.time() - start) >= time_limit:
                break

        avg = total / max(1, done)
        if avg > best_val:
            best_val = avg
            best_m = m

        if (time.time() - start) >= time_limit:
            break

    return best_m

# =========================
#   MCTS (Adversarial UCT)
# =========================
class Node:
    __slots__ = ("state", "parent", "move", "children", "untried", "visits", "value")
    def __init__(self, state: State, parent=None, move: Optional[Move]=None):
        self.state = state
        self.parent = parent
        self.move = move
        self.children: List["Node"] = []
        self.untried: List[Move] = legal_moves(state)
        self.visits = 0
        self.value = 0.0  # cumulative reward from AI perspective

    def uct_score(self, ch, c=1.4):
        if ch.visits == 0:
            return float("inf")
        exploit = ch.value / ch.visits
        explore = c * math.sqrt(math.log(self.visits + 1) / ch.visits)
        return exploit + explore

    def select_child_adversarial(self, c=1.4) -> "Node":
        """
        KEY FIX:
        - If it's AI's turn at this node -> choose child that MAXIMIZES UCT
        - If it's Human's turn -> choose child that MINIMIZES UCT
        This makes the search adversarial (not "helping AI").
        """
        if self.state.turn == AI_PLAYER:
            return max(self.children, key=lambda ch: self.uct_score(ch, c))
        else:
            return min(self.children, key=lambda ch: self.uct_score(ch, c))

    def add_child(self, move: Move, state: State) -> "Node":
        child = Node(state, parent=self, move=move)
        # remove one instance (safe even if duplicates not possible)
        for i, mm in enumerate(self.untried):
            if mm == move:
                self.untried.pop(i)
                break
        self.children.append(child)
        return child

    def update(self, reward: float):
        self.visits += 1
        self.value += reward

def mcts_best_move(root_state: State, time_limit=2.0, c=1.4) -> Move:
    root = Node(clone_state(root_state))
    start = time.time()

    while (time.time() - start) < time_limit:
        node = root
        state = clone_state(root_state)

        # 1) Selection
        while not node.untried and node.children:
            node = node.select_child_adversarial(c=c)
            state, _, _ = apply_move(state, node.move)

        # 2) Expansion
        if node.untried:
            # Slight bias: try immediate box moves first
            ims = immediate_box_moves(state)
            if ims:
                m = random.choice([mv for mv in ims if mv in node.untried] or node.untried)
            else:
                m = random.choice(node.untried)
            state, _, _ = apply_move(state, m)
            node = node.add_child(m, state)

        # 3) Simulation
        reward = rollout_playout(state)

        # 4) Backpropagation
        while node is not None:
            node.update(reward)
            node = node.parent

    # Choose child with most visits (robust)
    if not root.children:
        return random.choice(legal_moves(root_state))
    best_child = max(root.children, key=lambda ch: ch.visits)
    return best_child.move

# =========================
#           GUI
# =========================
class DotsBoxesGUI:
    def __init__(self, root):
        self.root = root
        self.root.title("Dots & Boxes 4x4 (Human vs AI) - Strong Monte Carlo / MCTS")
        self.state = new_state(first_player=HUMAN_PLAYER)
        self.ai_mode = tk.StringVar(value="MCTS")
        self.ai_busy = False

        # Top bar
        top = tk.Frame(root)
        top.pack(fill="x", padx=10, pady=6)

        tk.Label(top, text="AI Mode:").pack(side="left")
        self.mode_box = ttk.Combobox(
            top,
            textvariable=self.ai_mode,
            state="readonly",
            values=["Random", "MonteCarlo", "MCTS"],
            width=12
        )
        self.mode_box.pack(side="left", padx=6)

        tk.Label(top, text="AI Time (sec):").pack(side="left", padx=(12, 0))
        self.time_var = tk.DoubleVar(value=2.0)
        self.time_scale = ttk.Scale(top, from_=0.5, to=4.0, variable=self.time_var, orient="horizontal", length=160)
        self.time_scale.pack(side="left", padx=6)

        tk.Button(top, text="New Game", command=self.new_game).pack(side="left", padx=6)
        tk.Button(top, text="Rules", command=self.show_rules).pack(side="left", padx=6)

        self.info = tk.StringVar()
        tk.Label(top, textvariable=self.info).pack(side="right")

        # Canvas
        self.cell = 80
        self.pad = 40
        w = self.pad * 2 + self.cell * N
        h = self.pad * 2 + self.cell * N
        self.canvas = tk.Canvas(root, width=w, height=h, bg="white")
        self.canvas.pack(padx=10, pady=10)
        self.canvas.bind("<Button-1>", self.on_click)

        # Bottom status
        bottom = tk.Frame(root)
        bottom.pack(fill="x", padx=10, pady=6)
        self.status = tk.StringVar(value="Your turn (Blue). Click an empty edge.")
        tk.Label(bottom, textvariable=self.status, fg="blue").pack(side="left")
        self.score = tk.StringVar()
        tk.Label(bottom, textvariable=self.score, font=("Arial", 11, "bold")).pack(side="right")

        self.draw()
        self.update_labels()

    def show_rules(self):
        messagebox.showinfo(
            "Rules",
            "Dots & Boxes (4x4 boxes)\n\n"
            "• Click an empty edge to draw a line.\n"
            "• Completing a box gives 1 point AND an extra turn.\n"
            "• Game ends when all edges are filled.\n"
            "• Higher score wins.\n\n"
            "AI:\n"
            "• Random: random edges\n"
            "• MonteCarlo: evaluates moves via rollouts\n"
            "• MCTS: adversarial UCT + smart rollouts (strong)\n"
        )

    def new_game(self):
        self.state = new_state(first_player=HUMAN_PLAYER)
        self.ai_busy = False
        self.status.set("Your turn (Blue). Click an empty edge.")
        self.draw()
        self.update_labels()

    def update_labels(self):
        self.score.set(f"Human: {self.state.score1}   |   AI: {self.state.score2}")
        self.info.set(f"Turn: {'Human' if self.state.turn==HUMAN_PLAYER else 'AI'}")

    def end_if_terminal(self):
        if not is_terminal(self.state):
            return False

        if self.state.score1 > self.state.score2:
            messagebox.showinfo("Game Over", f"You win! {self.state.score1} - {self.state.score2}")
        elif self.state.score2 > self.state.score1:
            messagebox.showinfo("Game Over", f"AI wins! {self.state.score2} - {self.state.score1}")
        else:
            messagebox.showinfo("Game Over", f"Draw! {self.state.score1} - {self.state.score2}")
        return True

    def ai_take_turn(self):
        if self.ai_busy or self.state.turn != AI_PLAYER or is_terminal(self.state):
            return

        self.ai_busy = True
        self.status.set("AI thinking...")
        self.root.update_idletasks()

        t = float(self.time_var.get())
        mode = self.ai_mode.get()

        if mode == "Random":
            m = random.choice(legal_moves(self.state))
        elif mode == "MonteCarlo":
            m = best_move_montecarlo(self.state, sims_per_move=120, time_limit=t)
        else:
            m = mcts_best_move(self.state, time_limit=t, c=1.35)

        self.state, extra, gained = apply_move(self.state, m)

        if gained:
            self.status.set(f"AI got {gained} box(es) and plays again.")
        else:
            self.status.set("Your turn (Blue). Click an empty edge.")

        self.draw()
        self.update_labels()

        self.ai_busy = False

        if self.end_if_terminal():
            return

        if self.state.turn == AI_PLAYER:
            self.root.after(80, self.ai_take_turn)

    def on_click(self, event):
        if self.ai_busy or self.state.turn != HUMAN_PLAYER or is_terminal(self.state):
            return

        m = self.pick_move_from_click(event.x, event.y)
        if m is None:
            return
        if not self.is_move_empty(m):
            return

        self.state, extra, gained = apply_move(self.state, m)

        if gained:
            self.status.set(f"You got {gained} box(es)! Play again.")
        else:
            self.status.set("AI turn...")

        self.draw()
        self.update_labels()

        if self.end_if_terminal():
            return

        if self.state.turn == AI_PLAYER:
            self.root.after(120, self.ai_take_turn)

    def is_move_empty(self, m: Move) -> bool:
        if m.kind == 'H':
            return self.state.H[m.r][m.c] == 0
        return self.state.V[m.r][m.c] == 0

    # -------- Click -> Move detection --------
    def pick_move_from_click(self, x, y) -> Optional[Move]:
        tol = 12

        # Horizontal edges
        for r in range(N + 1):
            y0 = self.pad + r * self.cell
            for c in range(N):
                x1 = self.pad + c * self.cell
                x2 = self.pad + (c + 1) * self.cell
                if abs(y - y0) <= tol and (x1 + tol) <= x <= (x2 - tol):
                    return Move('H', r, c)

        # Vertical edges
        for r in range(N):
            y1 = self.pad + r * self.cell
            y2 = self.pad + (r + 1) * self.cell
            for c in range(N + 1):
                x0 = self.pad + c * self.cell
                if abs(x - x0) <= tol and (y1 + tol) <= y <= (y2 - tol):
                    return Move('V', r, c)

        return None

    # -------- Drawing --------
    def draw(self):
        self.canvas.delete("all")

        # Draw claimed boxes background
        for r in range(N):
            for c in range(N):
                owner = self.state.boxes[r][c]
                if owner == 0:
                    continue
                x1 = self.pad + c * self.cell
                y1 = self.pad + r * self.cell
                x2 = self.pad + (c + 1) * self.cell
                y2 = self.pad + (r + 1) * self.cell
                fill = "#cce5ff" if owner == HUMAN_PLAYER else "#ffd6cc"
                self.canvas.create_rectangle(x1 + 6, y1 + 6, x2 - 6, y2 - 6, fill=fill, outline="")

        # Draw edges (faint placeholders + colored claimed)
        for r in range(N + 1):
            y = self.pad + r * self.cell
            for c in range(N):
                x1 = self.pad + c * self.cell
                x2 = self.pad + (c + 1) * self.cell
                owner = self.state.H[r][c]
                if owner:
                    col = "blue" if owner == HUMAN_PLAYER else "red"
                    self.canvas.create_line(x1, y, x2, y, width=6, fill=col)
                else:
                    self.canvas.create_line(x1, y, x2, y, width=2, fill="#e6e6e6")

        for r in range(N):
            y1 = self.pad + r * self.cell
            y2 = self.pad + (r + 1) * self.cell
            for c in range(N + 1):
                x = self.pad + c * self.cell
                owner = self.state.V[r][c]
                if owner:
                    col = "blue" if owner == HUMAN_PLAYER else "red"
                    self.canvas.create_line(x, y1, x, y2, width=6, fill=col)
                else:
                    self.canvas.create_line(x, y1, x, y2, width=2, fill="#e6e6e6")

        # Draw dots
        for r in range(DOTS):
            for c in range(DOTS):
                x = self.pad + c * self.cell
                y = self.pad + r * self.cell
                self.canvas.create_oval(x - 5, y - 5, x + 5, y + 5, fill="black", outline="black")


if __name__ == "__main__":
    root = tk.Tk()
    app = DotsBoxesGUI(root)
    root.mainloop()
